# AFFECTA AI — RAF-DB Training (EXP-007 EffNet-B3 → EXP-008 ViT-Small)

Accuracy-first fine-tuning on RAF-DB official split with ImageNet-pretrained backbones.

- **GPU**: use a T4 x2 (or P100) accelerator from Run settings.
- **Data**: attach the public dataset `shuvoalok/raf-db-dataset` (Settings → Input) OR run the download cell below.
- **Outputs**: checkpoints + test_metrics land in `/kaggle/working/experiments/...` and get zipped to `/kaggle/working/affecta_results.zip`.

In [ ]:
import os, sys, subprocess
print('python', sys.version)
import torch
print('cuda available:', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 1. Install dependencies

Kaggle already ships torch + torchvision. We add timm (backbone zoo), plus pandas/sklearn which the trainer uses.

In [ ]:
!pip install -q timm pandas scikit-learn Pillow
import timm
print('timm', timm.__version__)

## 2. Get the training code

Clone the public repo so `ml/`, `src/`, `scripts/` are available. If you attach the GitHub repo instead (Settings → Input → GitHub), skip the clone.

In [ ]:
REPO = 'https://github.com/vizolabs/affecta-ai.git'
if not os.path.exists('affecta-ai'):
    !git clone --depth 1 {REPO}
# make project importable regardless of cwd
sys.path.insert(0, '/kaggle/working/affecta-ai')
os.chdir('/kaggle/working/affecta-ai')
print('repo files:', sorted(os.listdir('.'))[:12])

## 3. Data — RAF-DB official split

Two options (pick one):
1. **Attach as input** (recommended): Settings → Input → Datasets → add `shuvoalok/raf-db-dataset`. It appears at `/kaggle/input/raf-db-dataset`.
2. **Download in-cell** via the Kaggle API (works with the same Kaggle credentials used here) or a direct public zip.
Then run `scripts/prepare_rafdb.py --source kaggle` which reproduces the exact train/val/test layout the trainer expects.

In [ ]:
from pathlib import Path

raw = None
for r in ['/kaggle/input/raf-db-dataset', '/kaggle/input/shuvoalok/raf-db-dataset']:
    if (Path(r) / 'DATASET' / 'train').is_dir():
        raw = Path(r)
        break

if raw is None:
    print('Input dataset not attached; downloading via kaggle API...')
    !pip install -q kaggle
    !mkdir -p /kaggle/working/rafdb
    !kaggle datasets download -d shuvoalok/raf-db-dataset -p /kaggle/working/rafdb --unzip
    raw = Path('/kaggle/working/rafdb')
print('data root:', raw)
print('train dirs:', sorted((raw/'DATASET'/'train').iterdir())[:3] if (raw/'DATASET'/'train').is_dir() else 'MISSING')

In [ ]:
!python scripts/prepare_rafdb.py --raw {raw} --out /kaggle/working/processed/rafdb --source kaggle --val-fraction 0.15
print('---');
!cat /kaggle/working/processed/rafdb/summary.json

## 4. EXP-007 — EfficientNet-B3 (input 300, strong aug)

Command mirrors the local run pattern: staged fine-tune, ImageNet-pretrained, class-weighted CE + label smoothing 0.05, warm-start from timm pretrained weights (default `pretrained=True`).

In [ ]:
!PYTHONPATH=/kaggle/working/affecta-ai python -m ml.training.train_expression \
    --backbone efficientnet_b3 \
    --experiment-id EXP-007-EFFICIENTNET-B3 \
    --data-dir /kaggle/working/processed/rafdb \
    --epochs 30 --batch-size 32 --lr 1e-4 --weight-decay 1e-4 \
    --augmentation strong --amp
print('EXP-007 done')

In [ ]:
import json, glob
exp7 = glob.glob('/kaggle/working/affecta-ai/experiments/EXP-007*')[0]
print('EXP-007 dir:', exp7)
m = json.load(open(f'{exp7}/test_metrics.json'))
print('test_acc=%.2f%% test_macro_f1=%.2f%%' % (m['test_acc'], m['test_macro_f1']))

## 5. EXP-008 — ViT-Small (input 224, strong aug)

In [ ]:
!PYTHONPATH=/kaggle/working/affecta-ai python -m ml.training.train_expression \
    --backbone vit_small_patch16_224 \
    --experiment-id EXP-008-VIT-SMALL \
    --data-dir /kaggle/working/processed/rafdb \
    --epochs 30 --batch-size 32 --lr 1e-4 --weight-decay 1e-4 \
    --augmentation strong --amp
print('EXP-008 done')

In [ ]:
import json, glob
exp8 = glob.glob('/kaggle/working/affecta-ai/experiments/EXP-008*')[0]
print('EXP-008 dir:', exp8)
m = json.load(open(f'{exp8}/test_metrics.json'))
print('test_acc=%.2f%% test_macro_f1=%.2f%%' % (m['test_acc'], m['test_macro_f1']))

## 6. Package results for download

Zip the two experiment dirs (best.pt, test_metrics.json, train_history, config) so they can be pulled back and used for ensemble + TTA locally.

In [ ]:
import shutil, glob, os
out = '/kaggle/working/affecta_results'
os.makedirs(out, exist_ok=True)
for e in glob.glob('/kaggle/working/affecta-ai/experiments/EXP-00[78]*'):
    dst = out + '/' + os.path.basename(e)
    shutil.copytree(e, dst)
    print('copied', dst)
!cd /kaggle/working && zip -r -q affecta_results.zip affecta_results
!ls -lh /kaggle/working/affecta_results.zip

**Done.** Download `/kaggle/working/affecta_results.zip` (right panel → Files) and place it in the local project's `experiments/`.